# 실제 학습을 셀 단위로 실행
`local_paths.json`에 데이터·가중치 경로와 해시를 지정합니다. 기본 상태는 경로/recipe를 살펴보는 모드입니다. 준비 후 `LOAD_PRIVATE_INPUTS=True`로 바꾸어 위에서부터 실행합니다. 1 epoch 실행 셀은 별도 플래그로 제어합니다.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from the repository'
sys.path.insert(0, str(ROOT / 'src'))
import torch
torch.set_num_threads(2)
torch.manual_seed(20261112)
print('Repository', ROOT.name, '| PyTorch', torch.__version__)

## 1. 실험 선택

In [ ]:
import json
RECIPE = 'vits-supervised'  # or 'vitb-distillation'
DEVICE = 'cpu'             # GPU run: 'cuda'
LOAD_PRIVATE_INPUTS = False
RUN_ONE_EPOCH = False
config = json.loads((ROOT / f'configs/{RECIPE}.json').read_text())
print(config)

## 2. 데이터 쌍과 split 확인

In [ ]:
import hashlib
from cpdino_lab.data import discover_splits, make_batch, read_local_paths
if LOAD_PRIVATE_INPUTS:
    paths = read_local_paths(ROOT / 'local_paths.json')
    splits = discover_splits(paths['dataset_root'])
    manifest = []
    for split in ['train', 'valid']:
        for sample in splits[split]:
            for path in [sample.image, sample.flow]:
                with path.open('rb') as stream:
                    digest = hashlib.file_digest(stream, 'sha256').hexdigest()
                manifest.append((split, path.name, digest))
    dataset_hash = hashlib.sha256(json.dumps(manifest).encode()).hexdigest()
    print({key:len(value) for key,value in splits.items()})
else:
    print('Set local_paths.json and LOAD_PRIVATE_INPUTS to inspect private data')

## 3. 백본·헤드 생성과 검증된 가중치 초기화

In [ ]:
from cpdino_lab.models import build_model
from cpdino_lab.weights import load_weights
from cpdino_lab.checkpoints import save_checkpoint, restore_checkpoint
if LOAD_PRIVATE_INPUTS:
    student = build_model(config['student'])
    key = 'vits_encoder' if config['student']=='vits' else 'vitb_model'
    provenance = load_weights(student, paths[key]['path'], paths[key]['sha256'], scope=config['initialization_scope'])
    student.to(DEVICE)
    teacher = None
    if config['teacher']:
        teacher = build_model(config['teacher'])
        load_weights(teacher, paths['vitl_model']['path'], paths['vitl_model']['sha256'], scope='model')
        teacher.to(DEVICE).eval().requires_grad_(False)
    optimizer = torch.optim.AdamW([p for p in student.parameters() if p.requires_grad],
        lr=config['learning_rate'], weight_decay=config['weight_decay'])
    identity = {'recipe':config, 'student':provenance, 'dataset_sha256':dataset_hash,
                'teacher_sha256': paths['vitl_model']['sha256'] if teacher is not None else None}
    run_dir = ROOT / 'runs' / config['name']
    last_epoch = 0
    if (run_dir / 'last.pt').exists():
        last_epoch = restore_checkpoint(run_dir / 'last.pt', student, optimizer, identity)
    print('Ready', config['name'], 'at epoch', last_epoch)

## 4. 한 batch의 동작 확인
첫 1장을 사용해 업데이트를 확인합니다. 점검 전 상태를 저장하고 복원하여 이어지는 epoch 실행의 시작점을 보존합니다.

In [ ]:
from cpdino_lab.engine import train_step, evaluate
if LOAD_PRIVATE_INPUTS:
    save_checkpoint(run_dir/'inspection.pt', student, optimizer, last_epoch, identity)
    try:
        x, y = make_batch(splits['train'], [0], seed=42, device=DEVICE, tile_size=config['tile_size'])
        loss = train_step(student, optimizer, x, y, teacher, config['teacher_weight'])
        print('Batch shape', tuple(x.shape), '| loss', loss)
    finally:
        restore_checkpoint(run_dir/'inspection.pt', student, optimizer, identity)

## 5. 다음 1 epoch 학습·검증·저장
`RUN_ONE_EPOCH=True`로 바꾼 후 이 셀을 실행하면 다음 epoch 하나가 진행됩니다. 셀을 다시 실행하면 저장된 `last_epoch` 다음으로 진행합니다. 전체 목표는 recipe의 500 epoch이며 이 노트북은 epoch 단위 제어를 제공합니다.

In [ ]:
import numpy as np
if LOAD_PRIVATE_INPUTS and RUN_ONE_EPOCH:
    epoch = last_epoch + 1
    assert epoch <= config['epochs'], 'Recipe epoch budget already reached'
    order = np.random.default_rng(config['seed'] + epoch).permutation(len(splits['train']))
    weighted_loss = 0.0
    for offset in range(0, len(order), config['batch_size']):
        indices = order[offset:offset + config['batch_size']]
        x, y = make_batch(splits['train'], indices, epoch*1000+offset, DEVICE, config['tile_size'])
        value = train_step(student, optimizer, x, y, teacher, config['teacher_weight'])
        weighted_loss += value * len(indices)
    validation = None
    if epoch == 1 or epoch % config['validation_interval'] == 0:
        batches = (make_batch(splits['valid'], np.arange(offset, min(offset+config['batch_size'],len(splits['valid']))),
                   42+offset, DEVICE, config['tile_size'])
                   for offset in range(0, len(splits['valid']), config['batch_size']))
        validation = evaluate(student, batches)
    save_checkpoint(run_dir/'last.pt', student, optimizer, epoch, identity)
    last_epoch = epoch
    print({'epoch':epoch, 'train_loss':weighted_loss/len(order), 'validation_loss':validation})
else:
    print('One-epoch execution is controlled by LOAD_PRIVATE_INPUTS and RUN_ONE_EPOCH')

## 저장 형식과 기존 실험
이 노트북의 재개는 이 저장소가 저장한 optimizer/RNG 형식을 사용합니다. 원본 `best.pt`/raw model 가중치는 `load_weights(scope="model")`로 불러올 수 있습니다. 과거의 전체 학습 결과는 `03_results.ipynb`에서 확인합니다. 새 학습 결과는 별도 run으로 기록해야 합니다.